In [ ]:
import os
import pandas as pd
from datetime import date, datetime, time, timedelta
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

def interpolate_wait_times(ride_name, park_name, file_path):
  park_hours = pd.read_csv(f"{file_path}{park_name}_hours.csv")
  wait_times = pd.read_csv(f"{file_path}Wait_Times/{ride_name}.csv")

  wait_times["datetime"] = pd.to_datetime(wait_times["date"] + " " + wait_times["time"])
  park_hours["date"] = pd.to_datetime(park_hours["date"]).dt.date
  wait_times["start_date"] = pd.to_datetime(wait_times["start_date"]).dt.date
  wait_times["date"] = wait_times["datetime"].dt.date

  wait_times = wait_times.merge(park_hours, left_on="start_date", right_on="date", how="left")

  result = pd.DataFrame(columns=["date", "date_idx", "datetime_idx", "time", "wait_time", "minutes_from_open", "minutes_to_close", "day_of_week", "has_ea"])

  for date, group in wait_times.groupby("start_date"):

    open_dt = pd.to_datetime(f"{date} {group['open_time'].iloc[0]}")
    close_dt = pd.to_datetime(f"{date} {group['close_time'].iloc[0]}")

    ea = pd.notna(group['ea_time'].iloc[0])

    full_index = pd.date_range(start=pd.to_datetime(f"{date} 7:00:00"), end=pd.to_datetime(f"{date} 23:59:59"), freq="1min")

    group["datetime"] = group["datetime"].dt.round("min")
    group = group.sort_values("datetime").drop_duplicates("datetime", keep="last").set_index("datetime").sort_index()

    group = group.reindex(full_index)

    group["wait_time"] = group["wait_time"].interpolate(method="time", limit_area="inside")
    group["wait_time"] = group["wait_time"].fillna(0)


    group = group.reset_index(names="datetime")

    minutes_from_open = (group["datetime"] - open_dt).dt.total_seconds() / 60
    minutes_to_close = (close_dt - group["datetime"]).dt.total_seconds() / 60

    group["date"] = date
    group["date_idx"] = date
    group["datetime_idx"] = group["datetime"]
    group["time"] = group["datetime"].dt.time
    group["minutes_from_open"] = minutes_from_open
    group["minutes_to_close"] = minutes_to_close
    group["day_of_week"] = group["day_of_week"].ffill().bfill()
    group["has_ea"] = ea

    result = pd.concat([result, group[["date", "date_idx", "datetime_idx", "time", "wait_time", "minutes_from_open", "minutes_to_close", "day_of_week", "has_ea"]]], ignore_index=True)

  return result

def calculate_lag_rolling_stats(df):
  df["datetime"] = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str))
  df["date"] = pd.to_datetime(df["date"])

  df.set_index("datetime", inplace=True)

  df["wait_time_lag_day"] = df["wait_time"].shift(periods=1, freq="D", fill_value=0)
  df["wait_time_lag_week"] = df["wait_time"].shift(periods=7, freq="D", fill_value=0)
  df["wait_time_lag_year"] = df["wait_time"].shift(periods=365, freq="D", fill_value=0)

  df["wait_time_lag_day"] = df["wait_time_lag_day"].fillna(0)
  df["wait_time_lag_week"] = df["wait_time_lag_week"].fillna(0)
  df["wait_time_lag_year"] = df["wait_time_lag_year"].fillna(0)

  daily_avg = df[df["wait_time"] > 0].groupby("date")["wait_time"].mean()
  daily_max = df[df["wait_time"] > 0].groupby("date")["wait_time"].max()
  daily_std = df[df["wait_time"] > 0].groupby("date")["wait_time"].std()

  rolling_week_avg = daily_avg.shift(periods=1, freq="D", fill_value=0).rolling("7D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_week_avg"})
  rolling_month_avg = daily_avg.shift(periods=1, freq="D", fill_value=0).rolling("30D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_month_avg"})
  rolling_year_avg = daily_avg.shift(periods=1, freq="D", fill_value=0).rolling("365D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_year_avg"})

  df = df.merge(rolling_week_avg, on="date")
  df = df.merge(rolling_month_avg, on="date")
  df = df.merge(rolling_year_avg, on="date")

  rolling_week_max = daily_max.shift(periods=1, freq="D", fill_value=0).rolling("7D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_week_max"})
  rolling_month_max = daily_max.shift(periods=1, freq="D", fill_value=0).rolling("30D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_month_max"})
  rolling_year_max = daily_max.shift(periods=1, freq="D", fill_value=0).rolling("365D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_year_max"})

  df = df.merge(rolling_week_max, on="date")
  df = df.merge(rolling_month_max, on="date")
  df = df.merge(rolling_year_max, on="date")

  rolling_week_std = daily_std.shift(periods=1, freq="D", fill_value=0).rolling("7D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_week_std"})
  rolling_month_std = daily_std.shift(periods=1, freq="D", fill_value=0).rolling("30D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_month_std"})
  rolling_year_std = daily_std.shift(periods=1, freq="D", fill_value=0).rolling("365D").mean().reset_index().rename(columns={"wait_time": "wt_rolling_year_std"})

  df = df.merge(rolling_week_std, on="date")
  df = df.merge(rolling_month_std, on="date")
  df = df.merge(rolling_year_std, on="date")

  return df

def merge_ride_info(df, ride_name, park_name, file_path):
  ride_info = pd.read_csv(f"{file_path}{park_name}_ride_info.csv")

  return df.merge(ride_info[ride_info["ride_name"] == ride_name], how="cross")

def merge_weather_data(df, file_path):
  weather_data = pd.read_csv(f"{"/".join(file_path.split("/")[:-2])}/orlando_hourly_weather_data.csv")

  weather_data["datetime"] = pd.to_datetime(weather_data["date"] + " " + weather_data["hour"])
  df["datetime"] = pd.to_datetime(df["date"].astype(str) + " " + pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str)).dt.hour.astype(str) + ":00")

  return df.merge(weather_data.set_index("datetime")[["temp", "rhum", "prcp", "wspd", "pres", "cldc", "coco"]].shift(1).reset_index(), on="datetime", how="left").drop(columns="datetime")

def merge_event_holiday_info(df, file_path):
  event_holiday_info = pd.read_csv(f"{"/".join(file_path.split("/")[:-2])}/universal_event_holiday_info.csv")

  event_holiday_info["date"] = pd.to_datetime(event_holiday_info["date"])

  return df.merge(event_holiday_info, on="date", how="left")

def encode_date_time_data(df):
  df["date"] = pd.to_datetime(df["date"])

  df["year"] = df["date"].dt.year

  df["month_sin"] = df["date"].apply(lambda x : np.sin((2 * np.pi) * (x.month / 12)))
  df["month_cos"] = df["date"].apply(lambda x : np.cos((2 * np.pi) * (x.month / 12)))

  df["day_sin"] = df["date"].apply(lambda x : np.sin((2 * np.pi) * (x.day / 365)))
  df["day_cos"] = df["date"].apply(lambda x : np.cos((2 * np.pi) * (x.day / 365)))

  df["day_sin"] = df["date"].apply(lambda x : np.sin((2 * np.pi) * (x.day / 365)))
  df["day_cos"] = df["date"].apply(lambda x : np.cos((2 * np.pi) * (x.day / 365)))

  df["time_minutes"] = df["time"].apply(lambda x : (x.hour * 480) + x.minute)

  df["time_sin"] = df["time_minutes"].apply(lambda x : np.sin((2 * np.pi) * (x / 365)))
  df["time_cos"] = df["time_minutes"].apply(lambda x : np.cos((2 * np.pi) * (x / 365)))

  df["day_of_week_sin"] = df["day_of_week"].apply(lambda x : np.sin((2 * np.pi) * (x / 7)))
  df["day_of_week_cos"] = df["day_of_week"].apply(lambda x : np.cos((2 * np.pi) * (x / 7)))

  return df.drop(columns=["date", "time", "time_minutes", "day_of_week"])

def get_wait_time_features(park_name, file_path):
  for file_name in os.listdir(f"{file_path}Wait_Times/"):
    ride_name = file_name.replace(".csv", "")

    df = interpolate_wait_times(ride_name, park_name, file_path)
    df = calculate_lag_rolling_stats(df)
    df = merge_ride_info(df, ride_name, park_name, file_path).drop(columns="ride_name")
    df = merge_weather_data(df, file_path)
    df = merge_event_holiday_info(df, file_path)
    df = encode_date_time_data(df)

    print(f"Saving Features for {ride_name.replace("-", " ").title()}...\n")
    df.to_parquet(f"{file_path}Wait_Time_Features/{ride_name}_features.parquet", index=False)

def interpolate_baseline_preds(file_path):
  for file_name in os.listdir(f"{file_path}Baseline_Predictions/"):
    ride_name = file_name.replace(".csv", "")
    pred_wait_times = pd.read_csv(f"{file_path}/Baseline_Predictions/{file_name}")
    pred_wait_times["datetime"] = pd.to_datetime(pred_wait_times["date"] + " " + pred_wait_times["time"])

    y_pred_baseline = pd.DataFrame(columns=["datetime_idx", "wait_time"])

    for date, group in pred_wait_times.groupby("date"):
        full_index = pd.date_range(start=pd.to_datetime(f"{date} 7:00:00"), end=pd.to_datetime(f"{date} 23:59:59"), freq="1min")

        group["datetime"] = group["datetime"].dt.round("min")
        group = group.sort_values("datetime").drop_duplicates("datetime", keep="last").set_index("datetime").sort_index()

        group = group.reindex(full_index)

        group["wait_time"] = group["wait_time"].interpolate(method="time", limit_area="inside")
        group["wait_time"] = group["wait_time"].fillna(0)

        group = group.reset_index(names="datetime")

        group["datetime_idx"] = group["datetime"]

        y_pred_baseline = pd.concat([y_pred_baseline, group[["datetime_idx", "wait_time"]]], ignore_index=True)

    y_pred_baseline.to_parquet(f"{file_path}Baseline_Predictions/{ride_name}.parquet")

In [ ]:
# Interpolate Islands of Adventure Wait Time Features
get_wait_time_features("islands-of-adventure", "/content/drive/MyDrive/Theme_Park_Data/Universal_IOA/")

# Interpolate Universal Studios Wait Time Features
get_wait_time_features("universal-studios-florida", "/content/drive/MyDrive/Theme_Park_Data/Universal_Studios_FL/")

# Interpolate Islands of Adventure Baseline Predictions
interpolate_baseline_preds("/content/drive/MyDrive/Theme_Park_Data/Universal_IOA/")

# Interpolate Universal Studios Baseline Predictions
interpolate_baseline_preds("/content/drive/MyDrive/Theme_Park_Data/Universal_Studios_FL/")

Saving Features for Dudley Dorights Ripsaw Falls...

